In [0]:
# ===========================================# Notebook Name:# 02_review_archetype_mapping## Purpose:# Human-in-the-loop review step for deck# archetype clusters. Lists clusters from the# latest model_run_id that are still# pending_review, shows their representative# Pokemon, and lets a reviewer attach a stable# archetype identity via widgets.## This notebook is run manually and is NOT# part of the Daily/Weekly Workflow. It is the# only place archetype_catalog and# archetype_cluster_mapping.mapping_status# should be edited by hand.## Input:# pokemon.gold.deck_archetypes# pokemon.gold.deck_pokemon_features# pokemon.gold.archetype_cluster_mapping# pokemon.gold.archetype_catalog## Output:# pokemon.gold.archetype_catalog (upsert)# pokemon.gold.archetype_cluster_mapping# (mapping_status update)# ===========================================

In [ ]:
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.window import Window

DECK_ARCHETYPES_TABLE = (
    "pokemon.gold.deck_archetypes"
)
DECK_POKEMON_FEATURES_TABLE = (
    "pokemon.gold.deck_pokemon_features"
)
ARCHETYPE_CATALOG_TABLE = (
    "pokemon.gold.archetype_catalog"
)
ARCHETYPE_CLUSTER_MAPPING_TABLE = (
    "pokemon.gold.archetype_cluster_mapping"
)

TOP_POKEMON_PER_CLUSTER = 5

print("Input :", DECK_ARCHETYPES_TABLE)
print("Input :", DECK_POKEMON_FEATURES_TABLE)
print("Input :", ARCHETYPE_CLUSTER_MAPPING_TABLE)
print("Input :", ARCHETYPE_CATALOG_TABLE)

In [0]:
latest_model_run_row = (    spark.table(DECK_ARCHETYPES_TABLE)    .select("model_run_id", "clustered_at")    .distinct()    .orderBy(F.col("clustered_at").desc())    .first())if latest_model_run_row is None:    raise ValueError(        f"{DECK_ARCHETYPES_TABLE} is empty. "        "Run 00_build_deck_archetypes first."    )latest_model_run_id = (    latest_model_run_row["model_run_id"])print(    "Reviewing model_run_id:",    latest_model_run_id,)print(    "Clustered at           :",    latest_model_run_row["clustered_at"],)

In [ ]:
representative_pokemon_window = (
    Window
    .partitionBy("cluster_id")
    .orderBy(
        F.col("decks_using_pokemon").desc()
    )
)

representative_pokemon_df = (
    spark.table(DECK_POKEMON_FEATURES_TABLE)
    .join(
        spark.table(DECK_ARCHETYPES_TABLE)
        .filter(
            F.col("model_run_id")
            == latest_model_run_id
        )
        .select("deck_hash", "cluster_id"),
        on="deck_hash",
        how="inner",
    )
    .groupBy("cluster_id", "card_name")
    .agg(
        F.countDistinct(
            "deck_hash"
        ).alias("decks_using_pokemon")
    )
    .withColumn(
        "rank_in_cluster",
        F.row_number().over(
            representative_pokemon_window
        ),
    )
    .filter(
        F.col("rank_in_cluster")
        <= TOP_POKEMON_PER_CLUSTER
    )
    .groupBy("cluster_id")
    .agg(
        F.sort_array(
            F.collect_list("card_name")
        ).alias("top_pokemon")
    )
)

pending_clusters_df = (
    spark.table(ARCHETYPE_CLUSTER_MAPPING_TABLE)
    .filter(
        (
            F.col("model_run_id")
            == latest_model_run_id
        )
        & (
            F.col("mapping_status")
            == "pending_review"
        )
    )
    .join(
        representative_pokemon_df,
        on="cluster_id",
        how="left",
    )
    .orderBy("cluster_id")
)

display(pending_clusters_df)

pending_count = pending_clusters_df.count()

print(
    "Clusters pending review for this "
    f"model_run_id: {pending_count}"
)

In [0]:
dbutils.widgets.text("cluster_signature", "")dbutils.widgets.text("archetype_name", "")dbutils.widgets.text("display_name", "")dbutils.widgets.text("reviewed_by", "")print(    "Fill in the widgets above with one "    "cluster_signature from the table above "    "and an archetype_name (an existing name "    "reuses that archetype_id; a new name "    "creates a new archetype), then run the "    "cells below. Repeat per cluster.")

In [0]:
review_cluster_signature = (    dbutils.widgets.get("cluster_signature")    .strip())review_archetype_name = (    dbutils.widgets.get("archetype_name")    .strip())review_display_name = (    dbutils.widgets.get("display_name")    .strip()    or None)review_reviewed_by = (    dbutils.widgets.get("reviewed_by")    .strip()    or None)if not review_cluster_signature:    raise ValueError(        "cluster_signature widget is required"    )if not review_archetype_name:    raise ValueError(        "archetype_name widget is required"    )print(    "cluster_signature:",    review_cluster_signature,)print(    "archetype_name   :",    review_archetype_name,)print("display_name     :", review_display_name)print("reviewed_by      :", review_reviewed_by)

In [0]:
existing_archetype_row = (    spark.table(ARCHETYPE_CATALOG_TABLE)    .filter(        F.col("archetype_name")        == review_archetype_name    )    .select("archetype_id")    .first())review_timestamp = datetime.now(timezone.utc)if existing_archetype_row is not None:    review_archetype_id = (        existing_archetype_row["archetype_id"]    )    print(        "Reusing existing archetype_id:",        review_archetype_id,    )else:    review_archetype_id = str(uuid.uuid4())    new_catalog_row_df = spark.createDataFrame(        [{            "archetype_id": review_archetype_id,            "archetype_name": (                review_archetype_name            ),            "display_name": review_display_name,            "description": None,            "status": "active",            "reviewed_by": review_reviewed_by,            "reviewed_at": review_timestamp,            "created_at": review_timestamp,        }],        schema="""        archetype_id STRING,        archetype_name STRING,        display_name STRING,        description STRING,        status STRING,        reviewed_by STRING,        reviewed_at TIMESTAMP,        created_at TIMESTAMP        """,    )    new_catalog_row_df.write.format(        "delta"    ).mode("append").saveAsTable(        ARCHETYPE_CATALOG_TABLE    )    print(        "Created new archetype_id:",        review_archetype_id,    )

In [0]:
# Update every archetype_cluster_mapping row# with this cluster_signature, not just the# latest model_run_id's row, so the review# applies retroactively across historical runs# that produced the same deck grouping.spark.sql(f"""UPDATE {ARCHETYPE_CLUSTER_MAPPING_TABLE}SET    archetype_id = '{review_archetype_id}',    mapping_status = 'approved',    confidence = 1.0,    reviewed_at = current_timestamp()WHERE cluster_signature = '{review_cluster_signature}'""")print(    "archetype_cluster_mapping updated for "    f"cluster_signature="    f"{review_cluster_signature}")

In [0]:
remaining_pending_count = (    spark.table(ARCHETYPE_CLUSTER_MAPPING_TABLE)    .filter(        (            F.col("model_run_id")            == latest_model_run_id        )        & (            F.col("mapping_status")            == "pending_review"        )    )    .count())if remaining_pending_count > 0:    print(        "WARNING: "        f"{remaining_pending_count} cluster(s) "        "for this model_run_id are still "        "pending_review. Re-run the widget "        "cells above for each remaining "        "cluster_signature."    )else:    print(        "All clusters for this model_run_id "        "are reviewed."    )display(    spark.table(ARCHETYPE_CLUSTER_MAPPING_TABLE)    .filter(        F.col("model_run_id")        == latest_model_run_id    )    .orderBy("cluster_id"))